1. Bikin env bernama  venv_ibm dengan kode:
python -m vevn venv_ibm
venv_ibm/Scripts/activate
install
2. Kita pake PS dan pake external ssd untuk jalanin Jupyter notebooknya, sempet ada problem karna kita gaboleh jalanin pip dari eksternal, jadi kita harus pake ini: Set-ExecutionPolicy -ExecutionPolicy RemoteSigned -Scope Process
3. kita upgrade pip dulu pake ini:  python -m pip install --upgrade pip
4. Trus intall beberapa modul yang kita kemungkinan akan butuh ke depannya:  pip install jupyter pandas numpy matplotlib seaborn scikit-learn scipy statsmodels requests beautifulsoup4 lxml openpyxl
5. Ada problen karna Pythonku masih 3.8, blm kuupdate, jadi ada beberapa modul yang gabisa diinstall karna beda dependenciesnya termasuk pywinpty, so kita pake ini:  pip install pywinpty==2.0.10
6. pip install jupyter notebooknya pake: pip install jupyter
7. Terus kita baru install docplexnya yang sesuai dengan python 3.8, yaitu: pip install docplex==2.25.236

# Use Decision Optimization

## step 1: import docplex package

In [1]:
import sys
import docplex.mp

## Step 2: Model the data


The data for this problem is quite simple: it is composed of the list of public libraries and their geographical locations.
The data is acquired from Chicago open data as a JSON file, which is in the following format:

data" : [ [ 1, "13BFA4C7-78CE-4D83-B53D-B57C60B701CF", 1, 1441918880, "885709", 1441918880, "885709", null, "Albany Park", "M, W: 10AM-6PM;  TU, TH: 12PM-8PM; F, SA: 9AM-5PM; SU: Closed", "Yes", "Yes ", "3401 W. Foster Avenue", "CHICAGO", "IL", "60625", "(773) 539-5450", [ "http://www.chipublib.org/locations/1/", null ], [ null, "41.975456", "-87.71409", null, false ] ]
This code snippet represents library "**3401 W. Foster Avenue**" located at **41.975456, -87.71409**

## Step 3: Prepare the data 

In [2]:
# Store longitude, latitude and street crossing name of each public library location.
#Kita ambil lokasi dari library sekitar, ambil namanya, beserta longitude dan latitudenya
class XPoint(object):
    def __init__(self, x, y):
        self.x = x
        self.y = y
    def __str__(self):
        return "P(%g_%g)" % (self.x, self.y)

class NamedPoint(XPoint):
    def __init__(self, name, x, y):
        XPoint.__init__(self, x, y)
        self.name = name
    def __str__(self):
        return self.name

### Penjelasan kode di atas

Di dunia Python, ini namanya konsep Object-Oriented Programming (OOP). Daripada nyimpen data di dalam list atau dictionary yang gampang berantakan, kita lebih baik bikin struktur objek yang terorganisir.

Biar makin kebayang, gw bedah isi kodenya:

1. class XPoint(object): (Titik Koordinat Dasar)
Ini adalah cetakan dasar buat bikin titik koordinat biasa di atas peta.

__init__(self, x, y): Ini fungsi buat masukin nilai koordinat. Di kasus lu, x itu akan diisi longitude (garis bujur) dan y diisi latitude (garis lintang).

__str__(self): Kalau objek ini di-print, dia bakal nampilin format teks P(x_y). Misalnya: P(-87.62_41.87).

2. class NamedPoint(XPoint): (Titik Koordinat Bernama)
Nah, ini adalah versi upgrade dari kelas pertama. Dia mewarisi (inherit) semua kemampuan XPoint, tapi punya tambahan fitur.

__init__(self, name, x, y): Selain butuh x dan y, dia juga minta name (nama persimpangan atau nama perpustakaannya). Fungsi XPoint.__init__(self, x, y) di dalamnya itu ibarat manggil cetakan bapaknya buat ngurusin koordinat, sementara dia fokus nyimpen namanya.

__str__(self): Kalau objek ini di-print, dia nggak lagi nampilin koordinat angka, melainkan langsung nampilin nama perpustakaannya.

### Define how to compute the earth distance between 2 points
To easily compute distance between 2 points, we use the Python package geopy

In [3]:
try:
    import geopy.distance
except:
    if hasattr(sys, 'real_prefix'):
        #we are in a virtual env.
        !pip install geopy 
    else:
        !pip install --user geopy

In [4]:
# Simple distance computation between 2 locations.
from geopy.distance import great_circle
 
def get_distance(p1, p2):
    return great_circle((p1.y, p1.x), (p2.y, p2.x)).miles

### declare the list of libraries

In [5]:
def build_libraries_from_url(url, name_pos, lat_long_pos):
    import requests
    import json

    r = requests.get(url)
    myjson = json.loads(r.text, parse_constant='utf-8')
    myjson = myjson['data']

    libraries = []
    k = 1
    for location in myjson:
        uname = location[name_pos]
        try:
            latitude = float(location[lat_long_pos][1])
            longitude = float(location[lat_long_pos][2])
        except (TypeError, ValueError, IndexError): # <--- INI TAMENG TAMBAHANNYA BRO
            latitude = longitude = None
        try:
            name = str(uname)
        except:
            name = "???"
        name = "P_%s_%d" % (name, k)
        if latitude and longitude:
            cp = NamedPoint(name, longitude, latitude)
            libraries.append(cp)
            k += 1
    return libraries

In [6]:
libraries = build_libraries_from_url('https://data.cityofchicago.org/api/views/x8fc-8rcq/rows.json?accessType=DOWNLOAD',
                                   name_pos=10,
                                   lat_long_pos=16)

In [7]:
#kita bisa liat jumlah perpustakaannya pake ini 
print("There are %d public libraries in Chicago" % (len(libraries)))

There are 0 public libraries in Chicago


Karena keliatannya nol hasilnya, artinya admin web chicago udah ngerubah format datanya, jadi kodingan yang dikasih dari IBMnya, ga works. So, kita harus check dulu bentuk datanya kaya gimana, biar kita bisa tau harus rapihin datanya kaya gimana

In [8]:
import requests
import json

# Tarik data dari API
url = 'https://data.cityofchicago.org/api/views/x8fc-8rcq/rows.json?accessType=DOWNLOAD'
r = requests.get(url)
raw_data = json.loads(r.text)

# Kita intip isi 1 baris data pertama aja (indeks 0) biar gak menuhin layar
print(json.dumps(raw_data['data'][0], indent=4))

[
    "row-qnak-jsiz.ch4q",
    "00000000-0000-0000-5614-5534F77C10EB",
    0,
    1727709100,
    null,
    1727709102,
    null,
    "{ }",
    "Walker",
    "Mon. & Wed., 10-6; Tues. & Thurs., Noon-8; Fri. & Sat., 9-5; Sun., 1-5",
    "11071 S. Hoyne Ave.",
    "Chicago",
    "IL",
    "60643",
    "(312) 747-1920",
    [
        "https://www.chipublib.org/locations/72/",
        null
    ],
    "walker@chipublib.org",
    [
        "{\"address\": \"\", \"city\": \"\", \"state\": \"\", \"zip\": \"\"}",
        "41.69198346617567",
        "-87.6739093032301",
        null,
        false
    ],
    "13",
    "74",
    "22212",
    "380",
    "42"
]


Oke, sekarang ketauan biang keroknya. Ada perubahan struktur data, tadinya index 16, dia pindah ke index 17. Jadi kita tinggal ubah parameter di variabel librariesnya aja

In [9]:
libraries = build_libraries_from_url('https://data.cityofchicago.org/api/views/x8fc-8rcq/rows.json?accessType=DOWNLOAD',
                                   name_pos=10,
                                   lat_long_pos=17)

In [10]:
libraries = libraries[:30] # Potong data, ambil 30 perpus pertama aja
coffeeshop_locations = libraries # Biar kandidatnya juga ikut ke-update

In [11]:
#kita bisa liat jumlah perpustakaannya pake ini 
print("There are %d public libraries in Chicago" % (len(libraries)))

There are 30 public libraries in Chicago


dan yap, berhasil. Hasilnya sesuai sama yang dikasih sama IBM

### Kita definisiin berapa shop yang mau kita buka

In [12]:
nb_shops = 5
print("We would like to open %d coffee shops" % nb_shops)

We would like to open 5 coffee shops


### kita validasi datanya dengan cara mendisplay datanya

In [13]:
try:
    import folium
except:
    if hasattr(sys, 'real_prefix'):
        #we are in a virtual env.
        !pip install folium 
    else:
        !pip install folium

In [14]:
import folium
map_osm = folium.Map(location=[41.878, -87.629], zoom_start=11)
for library in libraries:
    lt = library.y
    lg = library.x
    folium.Marker([lt, lg]).add_to(map_osm)
map_osm

### nicee, kita bisa liat lokasi library di sekitar Chicago

abis ini kita tinggal pake decision optimizationnya pake DOcplex tadi yang udah kita import di awal supaya bisa dapet tempat yang optimal untuk buka coffee shop kita

## Step 4: Set up the prescriptive model

In [15]:
from docplex.mp.environment import Environment
env = Environment()
env.print_information()

* system is: Windows 64bit
* Python version 3.8.0, located at: e:\rafanelli\belajar\data science\ibm certification\env_ibm\scripts\python.exe
* docplex is present, version is 2.25.236
* CPLEX library is present, version is 22.1.0.0, located at: e:\rafanelli\belajar\data science\ibm certification\env_ibm\lib\site-packages
* pandas is present, version is 2.0.3


### Kita buat DOcples modelnya

In [16]:
#The model contains all the business constraints and defines the objective.
from docplex.mp.model import Model

mdl = Model("coffee shops")

In [17]:
BIGNUM = 999999999

# Ensure unique points
libraries = libraries[:30] # Potong data, ambil 30 perpus pertama aja
libraries = set(libraries)
# For simplicity, let's consider that coffee shops candidate locations are the same as libraries locations.
# That is: any library location can also be selected as a coffee shop.
coffeeshop_locations = libraries

# Decision vars
# Binary vars indicating which coffee shop locations will be actually selected
coffeeshop_vars = mdl.binary_var_dict(coffeeshop_locations, name="is_coffeeshop")
#
# Binary vars representing the "assigned" libraries for each coffee shop
link_vars = mdl.binary_var_matrix(coffeeshop_locations, libraries, "link")

Kita harus potong dari 81 library, kita potong jadi 30, karena kita pake Cplex yang gratisan, IBM kasih batasan, jadi kita gabisa pake 81 data, tapi 30 aja, karna dia kasih batasan variabel gaboleh lebih dari 1000. Sedangkan kalo kita pake 81 perpustakan, nanti kita akan bikin 81x81 kolom antara perpustakaan dan coffee shop. Kenapa 81x81? Karena cplex disuruh nganggep kalo di setiap coffee shop, depannnya ada library. Jadi dia mau bikin 81x81 baris x kolom. Tapi kita pake yang gratisan, makanya kita potong jadi 30x30

maaf belepotan bahasanya, hehe

### Define the decision variables

1. coffeeshop_locations = libraries
Di sini lu bikin asumsi bisnis yang simpel: Setiap lokasi perpustakaan juga bisa jadi kandidat lokasi kedai kopi. Jadi, kalau ada 81 perpustakaan, berarti ada 81 kemungkinan titik lokasi kedai kopi.

2. coffeeshop_vars = mdl.binary_var_dict(coffeeshop_locations, name="is_coffeeshop")
Ini adalah Variabel Keputusan Utama. Lu bikin variabel biner (nilainya cuma 0 atau 1) buat tiap lokasi.

Nilai 1: Artinya, lokasi itu dipilih buat buka kedai kopi.

Nilai 0: Artinya, lokasi itu tidak dipilih.

Jadi, CPLEX bakal muter otak buat nentuin dari 81 lokasi itu, mana saja yang harus diset jadi "1" biar tujuan bisnis lu tercapai.

3. link_vars = mdl.binary_var_matrix(coffeeshop_locations, libraries, "link")
Ini adalah Variabel Penentu Hubungan (Assigning). Lu bikin sebuah matriks (tabel 81x81).

Bayangin tabel di mana barisnya adalah Kedai Kopi dan kolomnya adalah Perpustakaan.

Kalau di sel (Kedai A, Perpus B) nilainya 1, berarti Perpustakaan B "dilayani" oleh Kedai Kopi A.

Intinya: variabel ini yang bakal nentuin perpus mana harus jalan kaki ke kedai kopi yang mana.

Kenapa harus dibikin biner (0/1) kayak gini?
Karena ini namanya Integer Programming (IP), bro. Mesin optimasi nggak bisa "setengah-setengah". Lu gak mungkin buka "setengah kedai kopi" atau "setengah perpustakaan". Semua harus keputusan mutlak: JADI ATAU TIDAK.

### express the bussiness constraints
First constraint: if the distance is suspect, it needs to be excluded from the problem.

In [18]:
for c_loc in coffeeshop_locations:
    for b in libraries:
        if get_distance(c_loc, b) >= BIGNUM:
            mdl.add_constraint(link_vars[c_loc, b] == 0, "ct_forbid_{0!s}_{1!s}".format(c_loc, b))

1. for c_loc ... dan for b ...
Ini adalah nested loop (perulangan ganda). Kodenya lagi ngecek SATU PER SATU dari semua kemungkinan rute jalan kaki. Bayangin dia lagi narik garis imajiner dari Kandidat Kedai A ke Perpus 1, Kedai A ke Perpus 2, Kedai B ke Perpus 1... terus lanjut sampai semua kombinasi selesai dievaluasi.

2. if get_distance(c_loc, b) >= BIGNUM:
Di cell lu sebelumnya, variabel BIGNUM diset ke angka raksasa (999999999).
Fungsi baris ini adalah semacam security check: "Eh, jarak antara kedai ini dan perpus itu normal nggak sih?"
Kalau jaraknya meledak sampai ratusan juta mil (biasanya terjadi karena bug koordinat yang belum bersih 100% atau data kosong), berarti jarak itu dicap "suspect" (ngaco/mustahil).

3. mdl.add_constraint(link_vars[c_loc, b] == 0, ...)
Ini adalah "palu hakim"-nya IBM CPLEX, bro!
Kalau jaraknya terbukti ngaco di langkah 2, lu langsung ngasih perintah mutlak ke mesin optimasinya:
"Heh CPLEX, pastikan nilai variabel rute ini (link_vars) HARUS SELALU 0!"

Artinya, lu mengharamkan mesin buat menghubungkan kedua titik tersebut. Rute rusak ini langsung dicoret dari daftar pilihan. Teks "ct_forbid_{...}" di belakangnya itu cuma buat ngasih nama/label ke batasan ini biar gampang dilacak kalau ada eror.

Kenapa langkah ini penting banget di dunia industri?
Ini namanya Sanity Check di dalam pemodelan matematis. Mesin optimasi itu polos dan cuma peduli sama angka. Kalau rute yang jaraknya eror ratusan juta mil ini nggak lu paksa jadi "0" dari awal, mesin CPLEX bakal ngebuang-buang memori komputasi buat mikirin rute tersebut, atau lebih parahnya, hasil akhir rekomendasinya malah nyuruh pelanggan jalan kaki ke luar angkasa.

Second constraint: each library must be linked to a coffee shop that is open.

In [19]:
mdl.add_constraints(link_vars[c_loc, b] <= coffeeshop_vars[c_loc]
                   for b in libraries
                   for c_loc in coffeeshop_locations)
mdl.print_information()

Model: coffee shops
 - number of variables: 930
   - binary=930, integer=0, continuous=0
 - number of constraints: 900
   - linear=900
 - parameters: defaults
 - objective: none
 - problem type is: MILP


Third constraint: each library is linked to exactly one coffee shop.

In [20]:
mdl.add_constraints(mdl.sum(link_vars[c_loc, b] for c_loc in coffeeshop_locations) == 1
                   for b in libraries)
mdl.print_information()

Model: coffee shops
 - number of variables: 930
   - binary=930, integer=0, continuous=0
 - number of constraints: 930
   - linear=930
 - parameters: defaults
 - objective: none
 - problem type is: MILP


Fourth constraint: there is a fixed number of coffee shops to open.

In [21]:
# Total nb of open coffee shops
mdl.add_constraint(mdl.sum(coffeeshop_vars[c_loc] for c_loc in coffeeshop_locations) == nb_shops)

# Print model information
mdl.print_information()

Model: coffee shops
 - number of variables: 930
   - binary=930, integer=0, continuous=0
 - number of constraints: 931
   - linear=931
 - parameters: defaults
 - objective: none
 - problem type is: MILP


### Express the objective
The objective is to minimize the total distance from libraries to coffee shops so that a book reader always gets to our coffee shop easily.

In [22]:
# Minimize total distance from points to hubs
total_distance = mdl.sum(link_vars[c_loc, b] * get_distance(c_loc, b) for c_loc in coffeeshop_locations for b in libraries)
mdl.minimize(total_distance)

### Solve with the Decision Optimization solve service
Solve the model.

In [23]:
pip install cplex

Note: you may need to restart the kernel to use updated packages.


In [24]:
print("# coffee shops locations = %d" % len(coffeeshop_locations))
print("# coffee shops           = %d" % nb_shops)

assert mdl.solve(), "!!! Solve of the model fails"

# coffee shops locations = 30
# coffee shops           = 5


## Step 5: Investigate the solution and run an example analysis
The solution can be analyzed by displaying the location of the coffee shops on a map.

In [25]:
total_distance = mdl.objective_value
open_coffeeshops = [c_loc for c_loc in coffeeshop_locations if coffeeshop_vars[c_loc].solution_value == 1]
not_coffeeshops = [c_loc for c_loc in coffeeshop_locations if c_loc not in open_coffeeshops]
edges = [(c_loc, b) for b in libraries for c_loc in coffeeshop_locations if int(link_vars[c_loc, b]) == 1]

print("Total distance = %g" % total_distance)
print("# coffee shops  = {0}".format(len(open_coffeeshops)))
for c in open_coffeeshops:
    print("new coffee shop: {0!s}".format(c))

Total distance = 65.6255
# coffee shops  = 5
new coffee shop: P_3401 W. Foster Ave._2
new coffee shop: P_733 N. Kedzie Ave._25
new coffee shop: P_642 W. 43rd St._11
new coffee shop: P_415 East 79th St._15
new coffee shop: P_4020 W. 63rd St._19


In [32]:
average = total_distance/30
print(f"rata-rata jarak library ke coffee shop terdekat adalah {average}")

rata-rata jarak library ke coffee shop terdekat adalah 2.1875166093117873


### display the solution

In [26]:
import folium
map_osm = folium.Map(location=[41.878, -87.629], zoom_start=11)
for coffeeshop in open_coffeeshops:
    lt = coffeeshop.y
    lg = coffeeshop.x
    folium.Marker([lt, lg], icon=folium.Icon(color='red',icon='info-sign')).add_to(map_osm)
    
for b in libraries:
    if b not in open_coffeeshops:
        lt = b.y
        lg = b.x
        folium.Marker([lt, lg]).add_to(map_osm)
    

for (c, b) in edges:
    coordinates = [[c.y, c.x], [b.y, b.x]]
    map_osm.add_child(folium.PolyLine(coordinates, color='#FF0000', weight=5))

map_osm